# Этап 2. Чанкинг и эмбеддинги

In [1]:
DEVICE="mps"

In [2]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("intfloat/multilingual-e5-base", device=DEVICE)
tokenizer = encoder.tokenizer

/Users/a1234/yandex/2/practics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1111 03:52:38.028000 34912 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7660.86it/s]


In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from chonkie import RecursiveChunker, SentenceChunker, TokenChunker
from IPython.display import HTML
from joblib import Memory


In [4]:
memory = Memory("cache", verbose=0)

In [5]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
def show(*, title: str, df: pd.DataFrame):
    display(HTML(f"""
        <h3 style="
            text-align: center;
            padding-bottom: 8px;
            border-bottom: 1px solid #888;
        ">{title}</h3>
    """))
    display(df)

In [6]:
def metrics(
    got_docs: list[int],
    relevants: list[int],
) -> tuple[float, float, float]:
    """Вычисляет precision, recall и MRR на уровне документов."""

    relevant = set(relevants)
    ranked_docs = list(dict.fromkeys(got_docs))

    if not relevant:
        return float("nan"), float("nan"), float("nan")

    hits = [doc_id for doc_id in ranked_docs if doc_id in relevant]

    precision = len(hits) / len(ranked_docs) if ranked_docs else 0.0
    recall = len(hits) / len(relevant)

    mrr = 1 / (ranked_docs.index(hits[0]) + 1) if hits else 0.0

    return precision, recall, mrr

In [7]:
docs = [json.loads(d) for d in Path("docs_clean.jsonl").read_text(encoding="utf-8").splitlines()]
queries = json.loads(Path("filmoskop_kb.json").read_text(encoding="utf-8"))["queries"]
docs_df = pd.DataFrame(docs)
print(f"Документы: {len(docs)}. Вопросы: {len(queries)}")

Документы: 135. Вопросы: 66


In [8]:
# средняя статистика по размеру текста документа в токенах
tmp = docs_df.assign(token_size=docs_df["text"].map(lambda t: len(tokenizer(t, add_special_tokens=False)["input_ids"])))
tmp["token_size"].describe()

count    135.000000
mean     204.451852
std      106.199876
min       14.000000
25%      114.500000
50%      177.000000
75%      273.500000
max      471.000000
Name: token_size, dtype: float64

Для выбора стратегии чанкинга рассмотрим следующие стратегии:
1. `Doc` без нарезки на чанки, документ == чанк.
Это основано на анализе размеров текстов документов.<br>
**max_size=471** **p75=273** говорит о том что корпус данных состоит из небольших фрагментов. И возможно, в большинстве случаев документ содержит ответ на один конкретный вопрос, а доп разбиение привидёт к потери смысла итогового вектора и ухудшит поиск. <br>Минусами такого подхода, является что он не масштабируемый, работает только для этого конкретного корпуса данных, если в корпусе будут документы размером 1000+ токенов то это сильно скажется на расходе токенов при генерации ответа, как следствие вырастет латенси. А также пострадает сам поиск, из-за размытого смысла получившегося ембеддинга по такому документу.
2. `fixed_size` - фиксированные чанки. Фиксированный размер чанка + overlap
3. `sentence` - нарезка чанков по предложениям.
4. `recursive` - рекурсивная нарезка сначала по абзацам, потом по предложениям.

В качестве реализации стратегий чанкинга я выбрал библиотеку `chonkie`



### Чанкинг

In [9]:
CHUNKERS = {
    "fixed_100": TokenChunker(tokenizer=tokenizer, chunk_size=100, chunk_overlap=20),
    "sentence_128": SentenceChunker(tokenizer=tokenizer, chunk_size=128),
    "sentence_256": SentenceChunker(tokenizer=tokenizer, chunk_size=256),
    "recursive_128": RecursiveChunker(tokenizer=tokenizer, chunk_size=128),
    "recursive_256": RecursiveChunker(tokenizer=tokenizer, chunk_size=256),

}
base = docs_df[["doc_id", "title", "category", "text"]]


def chunk_corpus(strategy: str) -> pd.DataFrame:
    if strategy == "doc":
        out = base.assign(chunk_text=base["text"])
    else:
        chunker = CHUNKERS[strategy]
        out = base.assign(chunk_text=base["text"].map(lambda t: [c.text for c in chunker(t)])).explode("chunk_text")
    out = out.drop(columns="text").assign(strategy=strategy)
    out["chunk_idx"] = out.groupby("doc_id").cumcount()
    return out

all_chunks = pd.concat([chunk_corpus(s) for s in ["doc", *CHUNKERS]], ignore_index=True)
all_chunks["token_count"] = [
    len(ids) for ids in tokenizer(all_chunks["chunk_text"].tolist(), add_special_tokens=False)["input_ids"]
]

#### Метрики

In [10]:
# признаки разрыва смысла
# признак что чанк последний из всех чанков документа
is_last = all_chunks["chunk_idx"] == all_chunks.groupby(["strategy", "doc_id"])["chunk_idx"].transform("max")
# оборвано посреди предложения
all_chunks["bad_end"] = ~is_last & ~all_chunks["chunk_text"].str.contains(r"[.!?…»:]\s*$")
# начинается с середины
all_chunks["lower_start"] = (all_chunks["chunk_idx"] > 0) & all_chunks["chunk_text"].str.match(r"[а-яё]")


chunk_stats = all_chunks.groupby("strategy").agg(
    chunks=("chunk_idx", "size"),
    tokens_min=("token_count", "min"),
    tokens_median=("token_count", "median"),
    tokens_max=("token_count", "max"),
    tokens_total=("token_count", "sum"),
    short_lt30=("token_count", lambda s: (s < 30).mean()),
    bad_end=("bad_end", "mean"),
    lower_start=("lower_start", "mean"),
)
chunk_stats["dup_factor"] = chunk_stats["tokens_total"] / chunk_stats.loc["doc", "tokens_total"]
show(title="Мертрики чанков в разрезе стратегий", df=chunk_stats.round(2))

,chunks,tokens_min,tokens_median,tokens_max,tokens_total,short_lt30,bad_end,lower_start,dup_factor
strategy,,,,,,,,,
doc,135,14,177.0,471,27601,0.01,0.00,0.00,1.00
fixed_100,384,14,100.0,101,32630,0.07,0.62,0.55,1.18
recursive_128,310,8,95.0,128,27601,0.04,0.01,0.01,1.00
recursive_256,183,5,148.0,256,27601,0.05,0.01,0.00,1.00
sentence_128,286,13,108.5,128,27601,0.05,0.00,0.01,1.00
sentence_256,182,5,146.5,256,27601,0.07,0.01,0.00,1.00


In [11]:
@memory.cache
def encode(texts: list[str]) -> np.ndarray:
    """Нормализованные эмбеддинги; кэш на диске привязан к самим текстам."""
    return encoder.encode(texts, batch_size=32, normalize_embeddings=True)


In [12]:
# каждая стратегия считается без заголовка и с заголовком документа в начале чанка
variants = pd.concat([
    all_chunks.assign(with_title=False, embed_text="passage: " + all_chunks["chunk_text"]),
    all_chunks.assign(with_title=True, embed_text="passage: " + all_chunks["title"] + "\n" + all_chunks["chunk_text"]),
], ignore_index=True)

vecs = encode(variants["embed_text"].tolist())

q_df = pd.DataFrame(queries)
q_vecs = encoder.encode(("query: " + q_df["query"]).tolist(), normalize_embeddings=True)

In [13]:
# отфильтровываем только вопросы с релевантными ответами
answerable = q_df[q_df["relevant_docs"].str.len() > 0]
relevant = answerable["relevant_docs"].map(set)


def evaluate(chunks_g: pd.DataFrame, vecs_g: np.ndarray, k: int = 5) -> pd.DataFrame:
    """
    цель метрик выяснить как стратегия чанкинга влияет на результаты поиска
    логика работы:
    есть 2 стратегии создания ембеддинга:  с включением в ембеддинг поля title и без
    оптимизируем работу, сравнения всех запросов со всем набором векторов документов
    выпоняем 1 запросом
    отбор запросов по индексам dataframe работает потому что индекс
    вектора в масиве ембеддингов запросов == индексу вопроса в dataframe
    нужно учесть что это работает если dataframe не изменяется(сортируется) после создания
    для расчёта precision используем не парметр top_k, а количество найденных уникальных документов
    """
    scores = q_vecs[answerable.index] @ vecs_g.T
    top = np.argsort(-scores, axis=1)[:, :k]

    doc_ids = chunks_g["doc_id"].to_numpy()
    tokens = chunks_g["token_count"].to_numpy()

    rows = []

    for idx, row_indices in enumerate(top):
        got_docs = doc_ids[row_indices].tolist()
        relevants = relevant.iloc[idx]

        precision, recall, mrr = metrics(
            got_docs=got_docs,
            relevants=relevants,
        )

        rows.append({
            f"precision@{k}": precision,
            f"recall@{k}": recall,
            f"mrr@{k}": mrr,
            f"context_tokens@{k}": tokens[row_indices].sum(),
            "kind": answerable["kind"].iloc[idx],
            f"unique_docs@{k}": len(set(got_docs)),
        })

    return pd.DataFrame(rows)


per_query = pd.concat(
    {key: evaluate(g, vecs[g.index]) for key, g in variants.groupby(["strategy", "with_title"], sort=True)},
    names=["strategy", "with_title", "qi"],
).reset_index()

retrieval_table = per_query.groupby(["strategy", "with_title"], sort=False)[["recall@5", "precision@5","unique_docs@5", "context_tokens@5"]].mean().round(3)
show(title="Среднее по метрикам recall precision для всех запросов", df=retrieval_table)
show(title="Срез по типам запросов", df=per_query.pivot_table(index=["strategy", "with_title"], columns="kind", values="recall@5", sort=False).round(2))
show(title="Мертрики чанков в разрезе стратегий", df=chunk_stats.round(2))

recall@5  precision@5  unique_docs@5  \
strategy      with_title                                         
doc           False          0.812        0.214          5.000   
              True           0.777        0.207          5.000   
fixed_100     False          0.732        0.257          4.000   
              True           0.723        0.324          3.339   
recursive_128 False          0.714        0.240          4.268   
              True           0.714        0.274          3.643   
recursive_256 False          0.786        0.228          4.589   
              True           0.723        0.231          4.250   
sentence_128  False          0.768        0.256          4.196   
              True           0.750        0.278          3.732   
sentence_256  False          0.777        0.228          4.607   
              True           0.732        0.232          4.268   

                          context_tokens@5  
strategy      with_title                    
doc           False               1254.821  
              True                1330.339  
fixed_100     False                430.679  
              True                 426.839  
recursive_128 False                452.589  
              True                 445.143  
recursive_256 False                881.036  
              True                 880.929  
sentence_128  False                500.696  
              True                 494.857  
sentence_256  False                887.250  
              True                 896.589

kind                      exact  semantic  multi
strategy      with_title                        
doc           False        1.00      0.74   0.75
              True         1.00      0.66   0.75
fixed_100     False        0.97      0.62   0.69
              True         0.97      0.66   0.59
recursive_128 False        1.00      0.60   0.62
              True         0.97      0.64   0.59
recursive_256 False        1.00      0.74   0.66
              True         0.97      0.62   0.66
sentence_128  False        1.00      0.68   0.69
              True         0.97      0.70   0.62
sentence_256  False        0.97      0.72   0.69
              True         0.97      0.66   0.62

,chunks,tokens_min,tokens_median,tokens_max,tokens_total,short_lt30,bad_end,lower_start,dup_factor
strategy,,,,,,,,,
doc,135,14,177.0,471,27601,0.01,0.00,0.00,1.00
fixed_100,384,14,100.0,101,32630,0.07,0.62,0.55,1.18
recursive_128,310,8,95.0,128,27601,0.04,0.01,0.01,1.00
recursive_256,183,5,148.0,256,27601,0.05,0.01,0.00,1.00
sentence_128,286,13,108.5,128,27601,0.05,0.00,0.01,1.00
sentence_256,182,5,146.5,256,27601,0.07,0.01,0.00,1.00


## Отбор финалистов: метрики нарезки и поиска

Для сравнения использовались шесть стратегий: `doc`, `fixed_100`, `recursive_128`, `recursive_256`, `sentence_128` и `sentence_256`.

### Сравнение стратегий

| Strategy | Chunks | Median tokens | Bad end | Lower start | Dup factor | Recall@5 | Recall@5 with title | Context tokens@5 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| doc | 135 | 177.0 | 0% | 0% | 1.00 | 0.812 | 0.777 | 1255 |
| fixed_100 | 384 | 100.0 | 62% | 55% | 1.18 | 0.732 | 0.723 | 431 |
| recursive_128 | 310 | 95.0 | 1% | 1% | 1.00 | 0.714 | 0.714 | 453 |
| recursive_256 | 183 | 148.0 | 1% | 0% | 1.00 | 0.786 | 0.723 | 881 |
| sentence_128 | 286 | 108.5 | 0% | 1% | 1.00 | 0.768 | 0.750 | 501 |
| sentence_256 | 182 | 146.5 | 1% | 0% | 1.00 | 0.777 | 0.732 | 887 |

### Отбор финалистов: метрики нарезки и поиска

**Метрики поиска не дают однозначного победителя.** Я считал их на 56 запросах с ответом в базе: top-5 чанков → уникальные разделы → сравнение с `relevant_docs`. Recall@5 находится в диапазоне 0,714–0,812. Лучший результат у `doc` (0,812), но среди стратегий чанкинга лидируют `recursive_256` (0,786) и `sentence_256` (0,777). Разница между финалистами составляет всего 0,009, то есть примерно половину одного запроса. Срезы по типам запросов (exact, semantic, multi) также не показывают устойчивого преимущества одной стратегии.

Precision@5 между стратегиями напрямую не сравниваю: у чанковых стратегий в top-5 попадает меньше уникальных разделов (примерно 3,3–4,6 против 5 у `doc`). Поэтому рост precision может быть связан с уменьшением числа разных разделов, а не с улучшением поиска.

**Заголовок в чанке.** Гипотеза: заголовок документа в начале чанка помогает восстановить контекст и улучшить поиск. Эффект не подтвердился: Recall@5 меняется разнонаправленно, но ни для одной стратегии добавление заголовка не даёт улучшения. Кроме того, количество уникальных разделов в top-5 уменьшается. Заголовок не использую.

**Финалисты: `recursive_256` и `sentence_256`.** Поскольку метрики поиска практически не различают эти стратегии, отбирал кандидатов по качеству самой нарезки:

- у обеих стратегий около 1% чанков с неудачным окончанием, нет чанков с началом из середины текста, нет overlap (dup factor 1,00);
- количество чанков практически одинаковое: 183 у `recursive_256` и 182 у `sentence_256`;
- `fixed_100` отброшен: 62% чанков имеют неудачное окончание, 55% начинаются с середины текста;
- варианты с окном 128 отброшены: больше фрагментов (286–310) без преимущества в поиске;
- `doc` оставлен как контрольный вариант: он показывает Recall@5 0,812, но требует около 1255 токенов контекста против 881–887 у финалистов.

**Следующий этап — ручная проверка `recursive_256` и `sentence_256`, чтобы выбрать стратегию, которая лучше сохраняет структуру и смысл исходных документов.**

### Ручной просмотр

In [14]:
FINALISTS = ["sentence_256", "recursive_256"]

def handle_look(strategy: str):
    ch = all_chunks[all_chunks["strategy"] == strategy].copy()
    ch["next_text"] = ch.groupby("doc_id")["chunk_text"].shift(-1)  # следующий чанк того же документа
    multi_ids = ch.loc[ch["chunk_idx"] > 0, "doc_id"].unique()      # документы, которые реально нарезаны
    multi = ch[ch["doc_id"].isin(multi_ids)]

    # 1. конец чанка и начало следующего (10 случайных стыков)
    bounds = ch.dropna(subset=["next_text"]).sample(10, random_state=1)
    show(title=f"{strategy} - Не разорваны ли факты на границах: конец чанка и начало следующего (10 случайных стыков)", df=pd.DataFrame({
        "doc_id": bounds["doc_id"].to_numpy(),
        "конец чанка": bounds["chunk_text"].str[-100:].to_numpy(),
        "начало следующего": bounds["next_text"].str[:100].to_numpy(),
    }))

    # 2. Не потерялось ли подлежащее: чанки, которые начинаются с местоимения или с маленькой буквы
    PRONOUN = r"^(он|она|оно|они|это|этот|эта|эти|такой|также|поэтому|там|тогда)\b"
    starts = multi[multi["chunk_idx"] > 0]
    text = starts["chunk_text"]
    bad_start = starts[text.str.lower().str.match(PRONOUN) | text.str.match(r"[а-яё]")]
    show(title=f"{strategy} - Без подлежащего в начале: {len(bad_start)} из {len(starts)} чанков", df=bad_start[["doc_id", "title", "chunk_text"]])

    # 3. Сохраняется ли смысл без соседей: самые короткие чанки и несколько документов целиком
    show(title=f"{strategy} - Сохраняется ли смысл без соседей: самые короткие чанки и несколько документов целиком", df=multi.nsmallest(8, "token_count")[["doc_id", "title", "token_count", "chunk_text"]])

    for doc_id in multi["doc_id"].drop_duplicates().sample(3, random_state=1):
        display(multi[multi["doc_id"] == doc_id][["doc_id", "title", "chunk_idx", "chunk_text"]])

for strategy in FINALISTS:
    handle_look(strategy)

,doc_id,конец чанка,начало следующего
0,91,"дакция собирает на главной — загляните в её полки, там есть и «Уютный вечер», и подборки поспорить.","А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."
1,81,"равится» тоже стоит ставить не из вежливости: они меняют то, что сервис предложит в следующий раз.\n\n","Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."
2,161,"подписан во время просмотра, эта настройка не влияет: интерфейс и контент локализуются независимо.\n\n","Насколько подробно локализована карточка, зависит от того, что передал правообладатель: у части ката"
3,144,- у части контента нет прав на офлайн-просмотр\n- на одном устройстве может храниться до 20 загрузок\n,- скачанный файл доступен до 30 дней\n- при подключении к интернету срок продлевается автоматически\n\n
4,5,"описаны в разделе об отмене подписки, и до их окончания обращение всё равно останется в ожидании.\n\n","Где посмотреть сведения о прошлых списаниях в самом сервисе, описано в отдельном разделе справки об"
5,20,"\n\nСтрана определяется по подключению: сервис ориентируется на сеть, из которой вы вышли в интернет.","Поэтому каталог меняется после подключения к местному Wi-Fi или мобильной сети, а не в момент пересе"
6,110,"сли хотя бы одно звено не поддерживает требования, сервис автоматически снизит качество до Full HD.","Фильмы с поддержкой 4K отмечены в карточке значком «4K».\nПроверить, что именно воспроизводится сейча"
7,128,ехода виден в разделе «Подписка»: там появляются название нового тарифа и дата следующего списания.,"Если настройки показывают прежний тариф, обновите страницу или перезапустите приложение — данные под"
8,173,"список устройств и отвяжите незнакомые. 4. Проверьте, не изменились ли привязанные почта и телефон.\n","Если изменить пароль не удаётся — доступ уже потерян, — обратитесь в поддержку через форму на сайте."
9,104,"а живёт своей жизнью: все настройки, списания и уведомления остаются на стороне владельца аккаунта.",Управлять этой подпиской из своего аккаунта не получится.


,doc_id,title,chunk_text
825,20,Просмотр за границей,"Поэтому каталог меняется после подключения к местному Wi-Fi или мобильной сети, а не в момент пересечения границы.\n\nЕсли нужного фильма в каталоге страны пребывания нет, добавьте его в список «Моё» — карточка сохранится, и после возвращения фильм снова станет доступен. Заново искать его в каталоге не придётся. То же касается сериалов: отмеченные позиции никуда не пропадают из списка.\n\nКогда после возвращения домой каталог остался прежним, перезапустите приложение и проверьте, что подключение идёт через домашнюю сеть. Если это не помогает, напишите в поддержку и укажите страну, из которой заходите."
922,116,Часто задаваемые вопросы об оплате,"Это особенность отображения в банковском приложении, на сами операции она не влияет."
985,179,Как выбрать HDMI-кабель,"Такой кабель отработал своё: его дешевле заменить, чем искать причину в настройках.\n\nСтоит посмотреть и на сами разъёмы: у телевизора их несколько, и они не всегда равнозначны. Какой порт основной, написано в инструкции."


,doc_id,title,token_count,chunk_text
972,168,Первые дни в «Фильмоскопе»,5,Приятного просмотра.
908,104,Можно ли подарить подписку,12,Управлять этой подпиской из своего аккаунта не получится.
958,157,Что сервис знает о ваших просмотрах,19,"Как их посмотреть и что с ними можно сделать, описано в отдельном разделе справки."
887,81,"Как выбрать, что посмотреть",20,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."
922,116,Часто задаваемые вопросы об оплате,20,"Это особенность отображения в банковском приложении, на сами операции она не влияет."
841,35,Почему банк отклоняет платёж,23,"Порядок действий на стороне сервиса, когда списание не удалось, описан в отдельном разделе справки об оплате."
896,91,Смотрим вместе: идеи для киновечера,23,"А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."
862,56,4K и HDR: что это такое,25,"Что должно совпасть, чтобы фильм действительно пошёл в 4K HDR, — тема отдельного раздела справки."


,doc_id,title,chunk_idx,chunk_text
895,91,Смотрим вместе: идеи для киновечера,0,"Киновечер — это не «включить что-нибудь фоном», а маленькое событие, ради которого люди договариваются о времени.\n\nНачните с темы. Один вечер — одно настроение: нуар под дождь за окном, ретро-комедии, мюзиклы, «фильмы, которые все смотрели, а мы почему-то нет». Тема снимает главную беду совместного просмотра — получасовые переговоры у главного экрана, после которых смотреть уже поздно.\n\nДоговоритесь, кто выбирает. Очередь по кругу работает лучше голосования: сегодня выбираете вы, в следующий раз — гость, который весь вечер ворчал. Право на один вечер и право на одну ошибку.\n\nНе забудьте про антураж: приглушённый свет, плед, попкорн и нарезка на общей тарелке, телефоны экраном вниз. Титры — тоже часть вечера, под них хорошо начинается разговор о фильме.\n\nГотовые темы для общей компании редакция собирает на главной — загляните в её полки, там есть и «Уютный вечер», и подборки поспорить."
896,91,Смотрим вместе: идеи для киновечера,1,"А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."


,doc_id,title,chunk_idx,chunk_text
886,81,"Как выбрать, что посмотреть",0,"Каталог большой, и выбор фильма на вечер иногда занимает больше времени, чем сам вечер. Несколько способов сузить его.\n\nПолки на главном экране. Главная собрана из тематических полок: новинки, популярное у зрителей, подборки редакции и персональные рекомендации профиля. Полки редакции одинаковы для всех, персональные — свои у каждого профиля.\n\nЖанры. В каталоге фильмы и сериалы разложены по жанрам, а внутри жанра список сортируется по году выхода или по популярности. Способ рабочий, когда настроение понятно, а название — нет: «хочу комедию, но не свежую».\n\nКарточка фильма. В ней собрано то, по чему обычно и принимают решение: возрастной рейтинг, год, страна, продолжительность, жанры и короткое описание без спойлеров.\n\nСписок «Моё». Всё, что заинтересовало, но сегодня не подошло, складывайте туда — через неделю у вас будет собственная полка на вечер. Оценки «Нравится» и «Не нравится» тоже стоит ставить не из вежливости: они меняют то, что сервис предложит в следующий раз.\n\n"
887,81,"Как выбрать, что посмотреть",1,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."


,doc_id,title,chunk_idx,chunk_text
962,161,Языки и локализация каталога,0,"Каталог «Фильмоскопа» собран для русскоязычного зрителя, поэтому почти всё, что вы видите на экране, переведено.\n\nНазвания. Зарубежные фильмы и сериалы подписаны прокатным русским названием — тем, под которым картина выходила в России. Оригинальное название указано в карточке рядом с годом выпуска: по нему проще найти знакомую картину, если прокатное название с ней расходится. Поиск понимает оба варианта, а часто и написание латиницей.\n\nОписания и материалы. Синопсис, жанры, имена актёров и режиссёров редакция ведёт на русском. Постеры и кадры берутся из локализованных материалов правообладателя, если они есть, — иначе показываются оригинальные.\n\nЯзык интерфейса. Он задаётся в настройках профиля и отвечает только за подписи меню, кнопок и разделов. На состав каталога и на то, как фильм звучит и подписан во время просмотра, эта настройка не влияет: интерфейс и контент локализуются независимо.\n\n"
963,161,Языки и локализация каталога,1,"Насколько подробно локализована карточка, зависит от того, что передал правообладатель: у части каталога описания короткие, а постеры остаются оригинальными."


... (2 outputs left out)

,doc_id,title,token_count,chunk_text
1465,168,Первые дни в «Фильмоскопе»,5,Приятного просмотра.
1379,81,"Как выбрать, что посмотреть",20,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."
1333,35,Почему банк отклоняет платёж,23,"Порядок действий на стороне сервиса, когда списание не удалось, описан в отдельном разделе справки об оплате."
1305,5,Финансовые вопросы: куда обратиться,26,"Где посмотреть сведения о прошлых списаниях в самом сервисе, описано в отдельном разделе справки об оплате."
1340,42,Один аккаунт на всю семью,26,"Если аккаунтов в семье почему-то оказалось несколько, что с ними делать — разобрано в отдельном разделе справки."
1371,72,Почему не удаётся войти в аккаунт,28,"Если пароль не подходит ни в одном варианте, дело уже не в опечатке — порядок восстановления описан в отдельном разделе справки."
1367,66,Приложения нет в магазине телевизора,29,"Поддерживается ли ваша модель, написано в разделе справки со списком устройств — начинать проще с него, а не с настроек."
1451,157,Что сервис знает о ваших просмотрах,29,"От зрителя эти записи не спрятаны. Как их посмотреть и что с ними можно сделать, описано в отдельном разделе справки."


,doc_id,title,chunk_idx,chunk_text
1387,91,Смотрим вместе: идеи для киновечера,0,"Киновечер — это не «включить что-нибудь фоном», а маленькое событие, ради которого люди договариваются о времени.\n\nНачните с темы. Один вечер — одно настроение: нуар под дождь за окном, ретро-комедии, мюзиклы, «фильмы, которые все смотрели, а мы почему-то нет». Тема снимает главную беду совместного просмотра — получасовые переговоры у главного экрана, после которых смотреть уже поздно.\n\nДоговоритесь, кто выбирает. Очередь по кругу работает лучше голосования: сегодня выбираете вы, в следующий раз — гость, который весь вечер ворчал. Право на один вечер и право на одну ошибку.\n\nНе забудьте про антураж: приглушённый свет, плед, попкорн и нарезка на общей тарелке, телефоны экраном вниз. Титры — тоже часть вечера, под них хорошо начинается разговор о фильме.\n\n"
1388,91,Смотрим вместе: идеи для киновечера,1,"Готовые темы для общей компании редакция собирает на главной — загляните в её полки, там есть и «Уютный вечер», и подборки поспорить. А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."


,doc_id,title,chunk_idx,chunk_text
1378,81,"Как выбрать, что посмотреть",0,"Каталог большой, и выбор фильма на вечер иногда занимает больше времени, чем сам вечер. Несколько способов сузить его.\n\nПолки на главном экране. Главная собрана из тематических полок: новинки, популярное у зрителей, подборки редакции и персональные рекомендации профиля. Полки редакции одинаковы для всех, персональные — свои у каждого профиля.\n\nЖанры. В каталоге фильмы и сериалы разложены по жанрам, а внутри жанра список сортируется по году выхода или по популярности. Способ рабочий, когда настроение понятно, а название — нет: «хочу комедию, но не свежую».\n\nКарточка фильма. В ней собрано то, по чему обычно и принимают решение: возрастной рейтинг, год, страна, продолжительность, жанры и короткое описание без спойлеров.\n\nСписок «Моё». Всё, что заинтересовало, но сегодня не подошло, складывайте туда — через неделю у вас будет собственная полка на вечер. Оценки «Нравится» и «Не нравится» тоже стоит ставить не из вежливости: они меняют то, что сервис предложит в следующий раз.\n\n"
1379,81,"Как выбрать, что посмотреть",1,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."


,doc_id,title,chunk_idx,chunk_text
1455,161,Языки и локализация каталога,0,"Каталог «Фильмоскопа» собран для русскоязычного зрителя, поэтому почти всё, что вы видите на экране, переведено.\n\nНазвания. Зарубежные фильмы и сериалы подписаны прокатным русским названием — тем, под которым картина выходила в России. Оригинальное название указано в карточке рядом с годом выпуска: по нему проще найти знакомую картину, если прокатное название с ней расходится. Поиск понимает оба варианта, а часто и написание латиницей.\n\nОписания и материалы. Синопсис, жанры, имена актёров и режиссёров редакция ведёт на русском. Постеры и кадры берутся из локализованных материалов правообладателя, если они есть, — иначе показываются оригинальные.\n\nЯзык интерфейса. Он задаётся в настройках профиля и отвечает только за подписи меню, кнопок и разделов. На состав каталога и на то, как фильм звучит и подписан во время просмотра, эта настройка не влияет: интерфейс и контент локализуются независимо.\n\n"
1456,161,Языки и локализация каталога,1,"Насколько подробно локализована карточка, зависит от того, что передал правообладатель: у части каталога описания короткие, а постеры остаются оригинальными."


## Выбор победителя: ручной просмотр финалистов

Для каждого финалиста я проверил три пункта: 10 случайных стыков соседних чанков (не разорваны ли связанные факты на границах), чанки без явно обозначенного подлежащего в начале и 8 самых коротких чанков. Дополнительно просмотрел три документа целиком (сохраняется ли смысл без соседей). Выборки у обеих нарезок одинаковые (`random_state=1`).

|                                    | **`recursive_256`**         | **`sentence_256`**                                |
| ---------------------------------- | --------------------------- | ------------------------------------------------- |
| Стыки на границе абзаца или строки | 9 из 10                     | 4 из 10                                            |
| Стыки внутри абзаца или списка     | 1 (список в doc 144)        | 6 (docs 91, 20, 110, 128, 104 и список в doc 144) |
| Чанки без подлежащего в начале     | 0 из 48                     | 3 из 47 (6%)                                      |
| Самые короткие чанки               | 5–29 токенов, преимущественно абзацы-ссылки | 5–25 токенов, в том числе оторванные концы абзацев |

**Границы.** `recursive_256` в 9 случаях из 10 режет по границам абзацев или строк. `sentence_256` чаще разрывает связанные части текста: в doc 20 объяснение определения страны отделено от последствий подключения к местной сети, в doc 128 информация о дате следующего списания отделена от инструкции, что делать, если отображается прежний тариф.

**Подлежащее.** У `recursive_256` таких чанков нет. У `sentence_256` три: «Поэтому каталог меняется…» (doc 20), «Это особенность отображения в банковском приложении…» (doc 116), «Такой кабель отработал своё…» (doc 179). Без предыдущего контекста начало этих фрагментов менее информативно.

**Смысл без соседей.** У обеих стратегий встречаются короткие абзацы-ссылки на другие разделы справки и отдельные фразы вроде «Приятного просмотра» (doc 168). Такие чанки сами по себе почти не несут полезной информации. Но у `sentence_256` дополнительно встречаются оторванные концы абзацев: например, в doc 104 фраза «Управлять этой подпиской из своего аккаунта не получится» отделена от предыдущего объяснения. В doc 91 отдельно оказался заключительный совет про совместный просмотр. В `recursive_256` эти части текста сохранены вместе. При просмотре документов целиком doc 81 и doc 161 нарезаны одинаково, а doc 91 лучше обработан `recursive_256`.

**Общее ограничение.** В обеих нарезках список в doc 144 разрезан между пунктами «до 20 загрузок» и «доступен до 30 дней».

**Победитель: `recursive_256`.** Поиск практически не различает финалистов: Recall@5 составляет 0,786 против 0,777, разница около половины запроса из 56. Объём контекста также почти одинаковый: 881 и 887 токенов. Решающее преимущество выявил ручной просмотр: `recursive_256` лучше сохраняет границы абзацев и связанные факты, не создаёт чанков без подлежащего и лучше сохраняет смысл фрагментов без соседнего контекста.

### Сохранение чанков и эмбеддингов

In [15]:
CHOSEN = "recursive_256"
MODEL_NAME = "intfloat/multilingual-e5-base"

# 1. Чанки победителя с метаданными и полным текстом
final = (
    all_chunks[all_chunks["strategy"] == CHOSEN]
    [["doc_id", "title", "category", "chunk_idx", "chunk_text", "token_count"]]
    .reset_index(drop=True)
)
final.insert(0, "chunk_id", final["doc_id"].astype(str) + ":" + final["chunk_idx"].astype(str))
final.to_json("chunks.jsonl", orient="records", lines=True, force_ascii=False)

# 2. Эмбеддинги считаем по сохранённому файлу
chunks_saved = pd.read_json("chunks.jsonl", lines=True)
chunk_vecs = encode(("passage: " + chunks_saved["chunk_text"]).tolist())
np.save("chunk_embeddings.npy", chunk_vecs)

# 3. Проверки: файл не исказил текст, строка i = вектор i, вектора те же, на которых считались метрики
assert chunks_saved["chunk_text"].tolist() == final["chunk_text"].tolist()
assert chunk_vecs.shape[0] == len(chunks_saved)
ref = vecs[variants.index[(variants["strategy"] == CHOSEN) & ~variants["with_title"]]]
assert np.allclose(chunk_vecs, ref, atol=1e-4)

# 4. Параметры, чтобы этап 3 строил запросы так же
Path("chunking_meta.json").write_text(json.dumps({
    "strategy": CHOSEN,
    "chunker": "chonkie.RecursiveChunker",
    "chunk_size_tokens": 256,
    "overlap": 0,
    "with_title": False,
    "embedding_model": MODEL_NAME,
    "passage_prefix": "passage: ",
    "query_prefix": "query: ",
    "normalized": True,
    "chunks": len(chunks_saved),
    "dim": int(chunk_vecs.shape[1]),
}, ensure_ascii=False, indent=2), encoding="utf-8")

show(title="Сохранённые чанки (первые 3)", df=chunks_saved.head(3))
show(title="Размеры", df=pd.Series({"chunks": len(chunks_saved), "embedding_shape": str(chunk_vecs.shape)}).to_frame("value"))

,chunk_id,doc_id,title,category,chunk_idx,chunk_text,token_count
0,0:0,0,Промокоды,Подписка и оплата,0,"Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки.\n\nПравила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.",113
1,1:0,1,Синхронизация между устройствами,Функции и контент,0,"Всё, что вы делаете в «Фильмоскопе», хранится не на устройстве, а в аккаунте. Поэтому телефон, планшет, браузер и телевизор показывают одно и то же: вошли — и сервис выглядит привычно, даже если устройство чужое.\n\nОбщее для аккаунта: подписка и тариф, способ оплаты, список привязанных устройств. Это не зависит ни от профиля, ни от того, откуда вы зашли.\n\nОбщее внутри профиля: список «Моё», оценки «Нравится» и «Не нравится», купленные фильмы и настройки самого профиля — язык интерфейса, автовоспроизведение, аватар. Отметили фильм на телефоне — он окажется в «Моём» и на телевизоре. При этом у каждого профиля всё своё: соседний профиль ваших оценок и вашего списка не увидит.\n\nОстаётся на устройстве: скачанные файлы и настройки, которые относятся к самой технике, — громкость, вывод звука, параметры изображения. Переносить их некуда, поэтому на новом устройстве они задаются заново.\n\n",236
2,1:1,1,Синхронизация между устройствами,Функции и контент,1,"Синхронизация требует интернета. Если устройство было офлайн, изменения доедут до аккаунта при следующем подключении, а пока экраны на разных устройствах могут расходиться.",40


,value
chunks,183
embedding_shape,"(183, 768)"


## Итог этапа

**Что сделал.**

Нарезал очищенный корпус из 135 документов шестью стратегиями, построил эмбеддинги `multilingual-e5-base` в двух вариантах: с заголовком документа и без него. Сравнил метрики нарезки и поиска, отобрал двух финалистов и провёл ручную проверку качества полученных чанков.

**Как выбирал.**

1. **Метрики поиска не выявили однозначного победителя.** Recall@5 стратегий находится в диапазоне 0,714–0,812. Среди финалистов `recursive_256` показывает 0,786, а `sentence_256` — 0,777. Разница составляет примерно половину запроса из 56, поэтому считать её существенным преимуществом нельзя. Добавление заголовка документа в эмбеддинг также не дало стабильного улучшения.

2. **По метрикам нарезки удалось отсеять менее удачные варианты.** `fixed_100` создаёт много фрагментов с неудачными границами: 62% чанков имеют признаки оборванного окончания, 55% начинаются с продолжения предыдущего текста. Стратегии с окном 128 создают больше фрагментов без заметного выигрыша в поиске. Вариант `doc` сохраняет документы целиком, но требует в среднем 1255 токенов контекста в top-5, тогда как у финалистов этот показатель составляет около 880 токенов. В финал вышли `recursive_256` и `sentence_256`.

3. **Ручной просмотр показал преимущество `recursive_256`.** В случайной выборке из 10 границ он сохранил границы абзацев или строк в 9 случаях, `sentence_256` — только в 4. Среди проверенных чанков `recursive_256` не было фрагментов без явно обозначенного подлежащего (0 из 48), тогда как у `sentence_256` таких оказалось 3 из 47. При просмотре документов целиком `recursive_256` также лучше сохранил связанные части текста.

Таким образом, одних метрик поиска оказалось недостаточно для выбора стратегии. Различия в Recall@5 между финалистами практически незначимы, поэтому решающим стало сочетание характеристик нарезки и результатов ручной проверки: сохранение смысловой целостности, качество границ и размер контекста.

**Выбор: `recursive_256` без заголовка.**

Использую `RecursiveChunker` из `chonkie` с размером чанка 256 токенов и без overlap. Стратегия последовательно старается сохранить структуру текста, разделяя его по абзацам, строкам и предложениям.

Результат: 183 чанка, медиана 148 токенов, максимальный размер 256 токенов. Все 27 601 токен исходного очищенного корпуса сохранены. Доля чанков с признаками неудачного окончания составляет около 1%, коэффициент дублирования — 1,00. В среднем в top-5 попадает 881 токен контекста.

**Ограничения.**

- 56 запросов позволяют выявить крупные различия, но недостаточны для уверенного сравнения близких результатов.
- В обеих финальных стратегиях список в doc 144 разрезан между чанками.
- Встречаются короткие чанки-ссылки и отдельные фразы длиной от 5 до 29 токенов, которые практически бесполезны без соседнего контекста.
- Recall считается на уровне документов и не показывает, насколько полно чанк сохраняет отдельный факт или инструкцию.
- Размер окна выбран на основании характеристик корпуса, а не исчерпывающего перебора. Для другого корпуса его потребуется пересмотреть.
- Короткие заглушки и архивные записи могут попадать в top-5 независимо от стратегии нарезки. К их фильтрации вернусь на этапе 4.
- Качество чанков непосредственно для генерации ответа дополнительно проверю на этапах 5–6.

**Результат этапа.**

Подготовлены артефакты для дальнейшей работы:

- `chunks.jsonl` — 183 чанка с полями `chunk_id`, `doc_id`, `title`, `category`, `chunk_idx`, `chunk_text`, `token_count`.
- `chunk_embeddings.npy` — матрица эмбеддингов размером `183 × 768`, нормализованных и рассчитанных с префиксом `passage: `.
- `chunking_meta.json` — параметры нарезки и используемой модели эмбеддингов.